# 04 - Model validation

Out-of-time validation of the champion PD model. Reads `artifacts/evaluation/*.json` produced by `make evaluate`.

In [1]:
import duckdb
import pandas as pd

from credit_risk.config import get_settings

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.4f}".format)
settings = get_settings()
con = duckdb.connect(str(settings.paths.warehouse), read_only=True)
q = lambda sql: con.execute(sql).df()  # noqa: E731
import json

ev = json.loads((settings.paths.evaluation / "evaluation.json").read_text())

In [2]:
print(ev["metadata"]["selection_rationale"])
pd.DataFrame(ev["diagnostics"])

Best challenger (random_forest) trails the logistic baseline by 0.0103 validation ROC-AUC (parsimony margin 0.01); the interpretable logistic baseline is kept.


,model,train_auc,validation_auc,test_auc,recent_auc,overfit_gap,temporal_change,diagnosis
0,hist_gradient_boosting,0.8532,0.7645,0.7818,0.7705,0.0887,0.0060,overfits (train >> validation)
1,logistic_regression,0.7849,0.7765,0.7840,0.7643,0.0084,-0.0122,generalises (small train-validation gap)
2,random_forest,0.8527,0.7662,0.7788,0.7677,0.0865,0.0015,overfits (train >> validation)


## Predicted risk vs actual outcomes (test window)

In [3]:
pd.DataFrame(ev["deciles_test"])[
    [
        "risk_decile",
        "population",
        "avg_predicted_pd",
        "actual_default_rate",
        "lift",
        "cum_bads_captured_pct",
    ]
]

,risk_decile,population,avg_predicted_pd,actual_default_rate,lift,cum_bads_captured_pct
0,1,220,0.4804,0.4682,3.2463,0.3249
1,2,220,0.2605,0.2818,1.9541,0.5205
2,3,220,0.1676,0.2045,1.4183,0.6625
3,4,219,0.1228,0.1233,0.8548,0.7476
4,5,220,0.0919,0.1227,0.8510,0.8328
5,6,220,0.0695,0.0773,0.5358,0.8864
6,7,219,0.0512,0.0868,0.6016,0.9464
7,8,220,0.0380,0.0364,0.2521,0.9716
8,9,220,0.0260,0.0273,0.1891,0.9905
9,10,220,0.0138,0.0136,0.0946,1.0000


## Calibration drift over time

In [4]:
q((settings.paths.sql / "analytics/17_actual_vs_predicted_by_month.sql").read_text()).tail(12)

,decision_month,scored_loans,avg_predicted_pd,matured_loans,actual_bad_rate,actual_to_predicted
12,2025-01-01,590,0.1158,590,0.0780,0.6730
13,2025-02-01,560,0.1266,560,0.1161,0.9170
14,2025-03-01,640,0.1217,640,0.1047,0.8600
15,2025-04-01,664,0.1330,664,0.1476,1.1100
16,2025-05-01,686,0.1243,686,0.1137,0.9150
17,2025-06-01,708,0.1275,708,0.1257,0.9860
18,2025-07-01,765,0.1279,765,0.1268,0.9920
19,2025-08-01,743,0.1352,743,0.1548,1.1450
20,2025-09-01,743,0.1330,690,0.1522,1.1370
21,2025-10-01,772,0.1334,656,0.1524,1.1220


## Why the latest months under-predict
Feature PSI is stable while the realised bad rate rises: the late-2025 stress raised the *level* of risk without moving the inputs much. That calls for recalibration on matured recent outcomes, not a rebuild.

In [5]:
mon = json.loads((settings.paths.evaluation / "monitoring.json").read_text())
pd.DataFrame(mon["performance"])

,window,matured_loans,bad_rate,mean_pd,calibration_ratio,roc_auc,pr_auc,brier,roc_auc_change_vs_train,pr_auc_change_vs_train
0,train,8813,0.1213,0.1213,0.9997,0.7849,0.3868,0.0903,0.0000,0.0000
1,validation,2058,0.1288,0.1282,0.9957,0.7765,0.3946,0.0960,-0.0084,0.0078
2,test,2198,0.1442,0.1322,0.9167,0.7840,0.3916,0.1054,-0.0009,0.0047
3,recent,863,0.1599,0.1408,0.8804,0.7643,0.3929,0.1173,-0.0206,0.0061


## Responsible-modeling review

In [6]:
pd.DataFrame(ev["fairness_test"])[
    [
        "attribute",
        "group",
        "n",
        "bad_rate",
        "mean_pd",
        "calibration_ratio",
        "roc_auc",
        "flagged_rate",
    ]
]

,attribute,group,n,bad_rate,mean_pd,calibration_ratio,roc_auc,flagged_rate
0,gender,female,989,0.1355,0.1304,0.9621,0.7719,0.1951
1,gender,male,1156,0.1566,0.1350,0.8622,0.7923,0.1920
2,gender,undisclosed,53,0.0377,0.1060,2.8082,0.7647,0.0943
3,age_band,18-24,233,0.2275,0.2202,0.9681,0.7468,0.4163
4,age_band,25-34,897,0.1672,0.1253,0.7491,0.7758,0.1695
5,age_band,35-44,661,0.1210,0.1188,0.9816,0.8122,0.1604
6,age_band,45-54,305,0.0852,0.1295,1.5191,0.7395,0.1869
7,age_band,55+,102,0.0784,0.0872,1.1117,0.7726,0.0784
8,region,Central,469,0.1279,0.1232,0.9634,0.7625,0.1748
9,region,Coastal,259,0.1815,0.1511,0.8328,0.7573,0.2355
